# MPバルクの整合性診断 — Llama 3B / Gemma 4B
## 実行状態
本番未実行。既定の `MODE="synthetic"` はCPU上の小さな人工行列だけを使います。
本番モデルのダウンロード、Drive書込、W&B送信は既定では実行しません。

## 方法
保存済み分散・閾値を固定し、λ=s²/n ≤ 閾値の固有値群を総質量1に再正規化してMPと比較します。
KSはp値ではありません。小さいbulkKSだけで真のシグナル分離やPPL維持は証明できません。
DE分散1の条件は独自のMP上端+TW閾値を用いる補助解析です。

## 環境
VS CodeのGoogle Colab拡張でNotebookを開き、Colabカーネルへ接続できます。
最初はCPUで人工行列テスト。モデル実験はまずT4で少数行列を計測し、必要なときだけL4等を検討してください。
A100への自動切替はありません。ランタイムは手動で選択します。
Colabではリポジトリを `/content/RMT_utils` にclone/pullしてから以下を実行してください。
依存関係（必要な場合のみ別セルで実行）:
`%pip install -r /content/RMT_utils/notebooks/requirements-bulk.txt`
既存Llama/Gemmaノートは変更しません。


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, subprocess, tempfile
import numpy as np
import pandas as pd
import torch

REPO = next((p for p in [Path.cwd(), Path.cwd().parent, Path('/content/RMT_utils')]
             if (p/'bulk_diagnostics.py').exists()), None)
if REPO is None:
    raise RuntimeError('RMT_utilsをcloneし、リポジトリまたはnotebooksフォルダで開いてください')
sys.path.insert(0, str(REPO))
from bulk_diagnostics import load_metrics, run_diagnostics, SOURCE_RUNS, publish_wandb, resolve_analysis_model
from funcs1 import get_esd_metrics

MODE = 'synthetic'  # synthetic / probe / full
ALLOW_MODEL_DOWNLOAD = False
PUBLISH_WANDB = False
MODEL_KEY = 'llama'  # llama / gemma（1モデルずつ）
MODEL_REVISION = None  # 本番時にHugging Faceのcommit SHAを指定する
SOURCE_RUN_FOR_MODEL = None  # 上記2つから、configで対象モデルを確認して指定
SOURCE_CONFIRMED = False  # revision・dtype・保存元モデルを確認してTrue
SVD_DEVICE = 'cpu'  # T4を手動選択後、probeで 'cuda' を評価する
BETA = 0.1
PROBE_LAYERS = 2
DRIVE_DATA = Path('/content/drive/MyDrive/TUS/hashiguchi/data')
EXPERIMENT_TAG = 'bulk-ks-v1'
MODEL_SPECS = {
    'llama': ('meta-llama/Llama-3.2-3B', 'llama-3.2-3B_esd_metrics.pkl'),
    'gemma': ('google/gemma-3-4b-pt', 'Gemma3-4B_esd_metrics.pkl'),
}
assert MODE in {'synthetic', 'probe', 'full'}
print('Mode:', MODE, '| No GPU is allocated by this notebook.')


## 入力と認証
DriveはVS Codeの `Colab: Mount Google Drive to Server...` でマウントできます。
本番時はHugging Faceでモデル利用許可を取得し、`huggingface_hub.login()`で認証します。
W&Bは送信前に`wandb.login()`を実行してください。トークンをNotebookに直書きしないでください。
保存元run（対応関係はrunのconfigを表示して確認）:
- https://wandb.ai/ryoya-zushi1210-tokyo-university-of-science/LlaMa-RMT-Analysis/runs/jv0d6kiu
- https://wandb.ai/ryoya-zushi1210-tokyo-university-of-science/LlaMa-RMT-Analysis/runs/vyzq5p34
保存された閾値は変更しません。既存KSは `_saved` 列に保持します。


In [ ]:
# このセルはrunの小さなメタデータを読むだけ。モデルやGPUは取得しません。
INSPECT_SOURCE_RUNS = False
if INSPECT_SOURCE_RUNS:
    import wandb
    wandb.login()
    api = wandb.Api()
    for source_run in SOURCE_RUNS:
        r = api.run(source_run)
        print('Source run:', source_run, 'name:', r.name)
        print({k:v for k,v in r.config.items()
               if k in ('model_id','model_name','model','revision','torch_dtype')})


In [ ]:
if MODE == 'synthetic':
    torch.manual_seed(42)
    model = torch.nn.Sequential(torch.nn.Linear(48, 24, bias=False),
                                torch.nn.Linear(24, 16, bias=False))
    saved = get_esd_metrics(model)
    OUTPUT = Path(tempfile.mkdtemp(prefix='rmt-bulk-smoke-'))
    metadata = {'model_id': 'synthetic/tiny-linear', 'source': 'seed=42', 'synthetic': True}
else:
    if not ALLOW_MODEL_DOWNLOAD or not SOURCE_CONFIRMED or not MODEL_REVISION or SOURCE_RUN_FOR_MODEL not in SOURCE_RUNS:
        raise RuntimeError('本番未承認: MODEL_REVISION・保存元を確認して実行フラグを設定してください')
    model_id, filename = MODEL_SPECS[MODEL_KEY]
    source = DRIVE_DATA/filename
    saved = load_metrics(source)
    OUTPUT = DRIVE_DATA/'bulk_ks'/MODEL_KEY/(EXPERIMENT_TAG+'-'+MODE)
    metadata = {'model_id': model_id, 'model_revision': MODEL_REVISION,
                'weight_dtype': 'float16', 'source_file': filename,
                'source_sha256': hashlib.sha256(source.read_bytes()).hexdigest(),
                'source_run': SOURCE_RUN_FOR_MODEL, 'candidate_source_runs': SOURCE_RUNS, 'synthetic': False}
metadata['git_commit'] = subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip()
metadata['code_sha256'] = {f: hashlib.sha256((REPO/f).read_bytes()).hexdigest()
                           for f in ['funcs1.py','bulk_diagnostics.py']}
metadata['torch_version'] = torch.__version__
metadata['numpy_version'] = np.__version__
print('Input rows:', len(saved), '| output:', OUTPUT)


## モデル取得と解析
本番ではモデル全体をCPUメモリに置き、SVD対象の行列だけを指定deviceへ転送します。
SVD用の一時領域も必要です。probeはモデル全体を取得するため、本番許可後にのみ実施します。
float64のSVDはT4で必ず高速になるとは限りません。probeの所要時間とメモリを確認します。
Gemmaは保存済み行列名に完全一致する言語モデル階層を選びます。一意でない場合は停止します。


In [ ]:
if MODE != 'synthetic':
    from transformers import AutoModelForCausalLM, Gemma3ForConditionalGeneration
    cls = Gemma3ForConditionalGeneration if MODEL_KEY == 'gemma' else AutoModelForCausalLM
    loaded = cls.from_pretrained(model_id, revision=MODEL_REVISION,
                                 torch_dtype=torch.float16, device_map='cpu', low_cpu_mem_usage=True)
    model = resolve_analysis_model(loaded, saved.name)
    if SVD_DEVICE == 'cuda':
        if not torch.cuda.is_available():
            raise RuntimeError('CUDA GPU未接続。CPUへ黙って切り替えず停止します')
        print('GPU:', torch.cuda.get_device_name())
        torch.cuda.reset_peak_memory_stats()

results = run_diagnostics(model, saved, OUTPUT, metadata, device=SVD_DEVICE,
                          beta=BETA, limit=PROBE_LAYERS if MODE == 'probe' else None)
if SVD_DEVICE == 'cuda':
    print('Peak allocated GB:', torch.cuda.max_memory_allocated()/1e9)
display(results[['name','KS_preDE','bulkKS_preDE','KS_postDE','bulkKS_postDE',
                 'KS_postDE_1','bulkKS_DE_1']].head(12))


## 結果の比較
全体KS→bulkKSの低下だけではDE効果としません。同じ行列のpre/postを比較します。
除いた固有値の比率も併記し、モデル別・モジュール別に見ます。
人工行列モードの図は動作確認であり、Llama/Gemmaについての研究結果ではありません。


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, full, bulk, title in [
    (axes[0], 'KS_preDE', 'bulkKS_preDE', 'Before DE'),
    (axes[1], 'KS_postDE', 'bulkKS_postDE', 'After DE (BEMA)'),
    (axes[2], 'KS_postDE_1', 'bulkKS_DE_1', 'After DE (variance=1)'),
]:
    ax.scatter(results[full], results[bulk], s=24, alpha=.75, color='#2864a0')
    ax.plot([0,1],[0,1], '--', color='gray')
    ax.set(xlabel='Full KS', ylabel='Bulk KS', title=title, xlim=(0,1), ylim=(0,1))
fig.suptitle(('SYNTHETIC CHECK — ' if MODE=='synthetic' else MODEL_KEY+' — ')+f'n={len(results)} matrices')
fig.savefig(OUTPUT/'full_vs_bulk.png', dpi=140)
plt.show()
fig, ax = plt.subplots(figsize=(6,4), constrained_layout=True)
sc = ax.scatter(results.bulkKS_preDE, results.bulkKS_postDE,
                c=results.bulk_ratio_postDE, vmin=0, vmax=1, cmap='viridis')
ax.plot([0,1],[0,1], '--', color='gray')
ax.set(xlabel='Bulk KS before DE', ylabel='Bulk KS after DE', xlim=(0,1), ylim=(0,1),
       title='Paired matrices: below diagonal = smaller KS after DE')
fig.colorbar(sc, ax=ax, label='Bulk fraction after DE')
fig.savefig(OUTPUT/'paired_bulk.png', dpi=140)
plt.show()
summary = results.groupby('module_type')[['bulkKS_preDE','bulkKS_postDE','bulkKS_DE_1',
                                         'bulkKS_delta_post_minus_pre','bulk_ratio_postDE']].median()
summary.to_csv(OUTPUT/'module_medians.csv')
display(summary)
print('Matrices with finite paired KS:', results[['bulkKS_preDE','bulkKS_postDE']].dropna().shape[0])


## 保存
CSVとmetadataは解析中からDriveの指定フォルダに保存します（人工行列は一時フォルダのみ）。
W&Bは新規runにTableと全結果Artifactを追加します。既存runは更新しません。
再開は同じ設定・同じコード・同じ入力に限ります。設定変更時はEXPERIMENT_TAGを変更してください。


In [ ]:
if PUBLISH_WANDB:
    if MODE != 'full':
        raise RuntimeError('W&Bへの研究結果の送信はfullモードだけで行います')
    url = publish_wandb(results, OUTPUT, metadata, [SOURCE_RUN_FOR_MODEL])
    print('W&B:', url)
else:
    print('W&B送信なし。結果保存先:', OUTPUT)


## 解釈と次の段階
本番未実行のため、DEの有効性について結論はまだありません。
まずT4/CPUでprobeを実行してメモリ・速度・保存済みスペクトルとの一致を確認します。
その後fullを明示的に実行します。小さいbulkKSは閾値処理の整合性を支持する診断であり、
シグナルの真偽・TW誤検出率・PPL維持を保証するものではありません。
